# 第 11 讲：整数规划、0-1 规划与多目标优化

> 用枚举法实现小规模 0-1 规划，并展示多目标权衡。

## 课件导学

**任务情境**：0-1 规划常用于选址、项目选择和任务分配，本讲用枚举法讲清离散选择的结构。

**关键概念**

- 0-1 变量
- 预算约束
- 可行组合
- 多目标权衡
- Pareto 思想

**本讲产物**

- binary_solutions.csv
- multiobjective_tradeoff.csv
- multiobjective_tradeoff.png

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [Python itertools.product](https://docs.python.org/3/library/itertools.html#itertools.product)：理解枚举所有 0-1 组合的方法。
- [SciPy MILP 文档](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html)：了解更大规模整数规划的求解接口。
- [SciPy linprog 文档](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html)：对比连续规划和整数规划的差异。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-11"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
projects = pd.DataFrame({
    "project": ["P1", "P2", "P3", "P4", "P5"],
    "benefit": [16, 22, 18, 12, 20],
    "cost": [8, 11, 9, 6, 10],
    "risk": [5, 7, 4, 3, 6],
})
budget = 24
rows = []
for bits in itertools.product([0, 1], repeat=len(projects)):
    chosen = np.array(bits)
    cost = (chosen * projects["cost"]).sum()
    if cost <= budget:
        rows.append({
            "selection": "".join(map(str, bits)),
            "benefit": (chosen * projects["benefit"]).sum(),
            "cost": cost,
            "risk": (chosen * projects["risk"]).sum(),
        })
solutions = pd.DataFrame(rows)
best = solutions.sort_values(["benefit", "risk"], ascending=[False, True]).head(1)
solutions.to_csv(OUTPUT_ROOT / "binary_solutions.csv", index=False)
best

In [ ]:
weights = np.linspace(0, 1, 21)
frontier = []
for w in weights:
    tmp = solutions.copy()
    tmp["score"] = w * tmp["benefit"] - (1 - w) * tmp["risk"]
    row = tmp.sort_values("score", ascending=False).iloc[0]
    frontier.append({"benefit_weight": w, "selection": row["selection"], "benefit": row["benefit"], "risk": row["risk"]})
frontier = pd.DataFrame(frontier)
frontier.to_csv(OUTPUT_ROOT / "multiobjective_tradeoff.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(solutions["risk"], solutions["benefit"], alpha=0.5, label="feasible")
ax.plot(frontier["risk"], frontier["benefit"], "r-o", label="weighted choices")
ax.set_xlabel("Risk")
ax.set_ylabel("Benefit")
ax.set_title("Benefit-risk tradeoff")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "multiobjective_tradeoff.png", dpi=160)
frontier.head()

## 实战环节

**课堂任务**

- 修改预算上限，观察最优组合变化。
- 改变收益和风险权重，观察折中前沿。
- 给一个组合写出选择理由。

**课后挑战**：加入一个互斥约束，例如 P1 和 P3 不能同时选择。

**验收清单**

- 是否枚举所有可行组合
- 预算约束是否生效
- 是否说明多目标权衡

In [ ]:
lesson_resources = [{'title': 'Python itertools.product', 'url': 'https://docs.python.org/3/library/itertools.html#itertools.product', 'reading_note': '理解枚举所有 0-1 组合的方法。'}, {'title': 'SciPy MILP 文档', 'url': 'https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html', 'reading_note': '了解更大规模整数规划的求解接口。'}, {'title': 'SciPy linprog 文档', 'url': 'https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html', 'reading_note': '对比连续规划和整数规划的差异。'}]
lesson_deliverables = ['binary_solutions.csv', 'multiobjective_tradeoff.csv', 'multiobjective_tradeoff.png']
lesson_checklist = ['是否枚举所有可行组合', '预算约束是否生效', '是否说明多目标权衡']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")